# 深度域礁核相控半径调试 QC

用途：地质人员从地质软件取得礁核 `x/y/depth` 后，在这里反复调试横向半径 `radius_xy_m`、纵向半径 `radius_z_m` 和目标低阻抗 `target_ai`。

运行节奏：

- **只跑一次**：环境/路径配置、大体数据加载。
- **可反复跑**：礁核参数、局部相控计算、剖面 QC、正演 QC、保存 trial。

注意：半径单位均为米；`target_ai` 是绝对 AI 值，不是百分比；本 notebook 只做 QC 和调参记录，不生成正式生产 NPZ。正式多点产出请使用 `scripts/lfm_facies_control_depth.py`。

In [ ]:
# Cell 1 - 只跑一次：环境与路径配置
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    # 如果 notebook 当前目录是 notebooks/，则回到 repo root
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

COMMON_CONFIG = REPO_ROOT / 'experiments/common_depth.yaml'
TRAIN_CONFIG = REPO_ROOT / 'experiments/ginn_depth/train.yaml'

# 可按需覆盖；None 表示从 common_depth.yaml / train.yaml 自动读取
SOURCE_AI_LFM_FILE = None
VP_LFM_FILE = None
WAVELET_FILE = None
DYNAMIC_GAIN_FILE = None
SEISMIC_FILE = None

# 正演建议使用 cpu；单道 QC 不需要 GPU
FORWARD_DEVICE = 'cpu'

print('REPO_ROOT =', REPO_ROOT)
print('COMMON_CONFIG =', COMMON_CONFIG)
print('TRAIN_CONFIG =', TRAIN_CONFIG)

In [ ]:
# Cell 2 - 只跑一次：加载大体数据、层位、survey、wavelet、正演模型
from cup.seismic.facies_control_qc_depth import (
    append_trial_record,
    apply_single_control_to_window,
    forward_qc_at_control_trace,
    load_qc_context,
    make_control_point,
    plot_control_sections,
    plot_forward_qc,
    pretty_json,
    read_notebook_run_summary,
    trial_csv_path,
)

ctx = load_qc_context(
    common_config_path=COMMON_CONFIG,
    train_config_path=TRAIN_CONFIG,
    source_ai_lfm_file=SOURCE_AI_LFM_FILE,
    vp_lfm_file=VP_LFM_FILE,
    wavelet_file=WAVELET_FILE,
    dynamic_gain_file=DYNAMIC_GAIN_FILE,
    seismic_file=SEISMIC_FILE,
    device=FORWARD_DEVICE,
)

print(pretty_json(read_notebook_run_summary(ctx)))

## Cell 3 参数提醒

第一次运行前请先填写 `x/y/depth_m/target_ai`。模板不会提供默认坐标，避免把示例点误当成生产点。

In [ ]:
# Cell 3 - 可反复跑：礁核参数，只改这里
# 必须先从地质软件填入礁核中心坐标和深度；半径单位是米，target_ai 是绝对 AI。
# 默认 None 会主动报错，避免误用模板占位坐标。
control_name = 'reef_core_001'
x = None
y = None
depth_m = None
target_ai = None
radius_xy_m = 300.0
radius_z_m = 35.0
strength = 1.0

# 正演 influence_only 指标使用的权重阈值；raised-cosine 下 0.01 基本覆盖到透镜体边界。
influence_weight_threshold = 0.01

# 剖面显示范围：横向半宽约为 section_scale * 2 * radius_xy_m；纵向半高为 section_scale * radius_z_m
section_scale = 1.2

required_values = {
    'x': x,
    'y': y,
    'depth_m': depth_m,
    'target_ai': target_ai,
    'radius_xy_m': radius_xy_m,
    'radius_z_m': radius_z_m,
    'strength': strength,
}
missing = [name for name, value in required_values.items() if value is None]
if missing:
    raise ValueError('请先在 Cell 3 填写这些参数：' + ', '.join(missing))

point = make_control_point(
    name=control_name,
    x=x,
    y=y,
    depth_m=depth_m,
    radius_xy_m=radius_xy_m,
    radius_z_m=radius_z_m,
    target_ai=target_ai,
    strength=strength,
)
point

In [ ]:
# Cell 4 - 可反复跑：局部相控计算
# 只截取局部窗口，不处理整块体；横向窗口按真实 XY 米制距离选取。
result = apply_single_control_to_window(ctx, point, section_scale=section_scale)

print(f'inline={result.inline:.2f}, xline={result.xline:.2f}')
print(f'nearest indices: il={result.il_idx}, xl={result.xl_idx}')
if result.warning:
    print('WARNING:', result.warning)
display(result.qc_df.drop(columns=['horizon_values'], errors='ignore'))
print('horizon values at control point:')
print(pretty_json(result.qc_df.iloc[0]['horizon_values']))

In [ ]:
# Cell 5 - 可反复跑：地质体剖面 QC
# 展示过礁核的 inline/xline 两个方向剖面：before、after、差值、权重。
plot_control_sections(result)

In [ ]:
# Cell 6 - 可反复跑：礁核道正演 QC
# 同时展示物理幅度口径和局部 RMS 匹配口径；指标会分别给出 display_window 与 influence_only。
metrics, waveforms = forward_qc_at_control_trace(
    ctx,
    result,
    influence_weight_threshold=influence_weight_threshold,
)
plot_forward_qc(metrics, waveforms, gain_mode=ctx.gain_mode)

# 常看指标：corr 越高越好；nmae/rmse 越低越好；rms_ratio 接近 1 表示幅度更接近实际地震。
display(metrics)

In [ ]:
# Cell 7 - 可选反复跑：保存当前 trial，便于比较多组半径
output_csv = trial_csv_path(ctx.repo_root, control_name)
trial_df = append_trial_record(output_csv, result, metrics)
print('Saved trial CSV:', output_csv)

preferred_sort_cols = [
    'influence_only_physical_after_corr',
    'display_window_physical_after_corr',
    'physical_after_corr',  # 兼容旧 trial CSV
]
sort_col = next((col for col in preferred_sort_cols if col in trial_df.columns), None)
if sort_col is not None:
    display(trial_df.sort_values(sort_col, ascending=False).head(20))
else:
    display(trial_df.tail(20))